<a href="https://colab.research.google.com/github/abdullah-subial/dubai-nlp-engine/blob/main/Dubai_NLP_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Fetch Dubai Restaurants Reviews**



In [ ]:
import os
import time
import requests
import pandas as pd
import numpy as np


def _get_api_key() -> str:
    """Fetches the Google Places API key from Colab Secrets when available,
    falling back to the GOOGLE_PLACES_API_KEY environment variable everywhere else."""
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_PLACES_API_KEY")
        if key:
            return key
    except Exception:
        pass

    key = os.environ.get("GOOGLE_PLACES_API_KEY")
    if not key:
        raise RuntimeError(
            "GOOGLE_PLACES_API_KEY not found. Set it as a Colab secret or an "
            "environment variable before calling get_reviews_for_area()."
        )
    return key


API_KEY = _get_api_key()

PLACES_SEARCH_URL = "https://places.googleapis.com/v1/places:searchText"
FIELD_MASK = (
    "places.id,"
    "places.displayName,"
    "places.rating,"
    "places.userRatingCount,"     # Total Google ratings
    "places.location,"            # Lat/Lng for Heatmap
    "places.reviews,"             # Includes publishTime & text
    "places.priceLevel,"
    "places.priceRange,"
    "places.primaryTypeDisplayName,"
    "places.formattedAddress,"
    "places.shortFormattedAddress,"
    "nextPageToken"
)


def _fetch_places_pages(query_string: str, max_pages: int = 1, page_delay_seconds: float = 2.0) -> list:
    """
    Calls Google Places Text Search (New) and follows pagination up to `max_pages`
    (Google returns at most 20 places per page, up to 60 total across 3 pages).
    Defaults to a single page to keep the current API cost/latency profile;
    raise max_pages when an area needs deeper coverage.
    """
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": FIELD_MASK,
    }

    all_places = []
    page_token = None

    for _ in range(max_pages):
        payload = {"textQuery": query_string, "languageCode": "en", "regionCode": "AE"}
        if page_token:
            payload["pageToken"] = page_token

        try:
            response = requests.post(PLACES_SEARCH_URL, headers=headers, json=payload, timeout=15)
            response.raise_for_status()
        except requests.exceptions.RequestException as exc:
            raise RuntimeError(f"Google Places API request failed: {exc}") from exc

        data = response.json()
        all_places.extend(data.get("places", []))

        page_token = data.get("nextPageToken")
        if not page_token:
            break

        # Google requires a short delay before a nextPageToken becomes usable.
        time.sleep(page_delay_seconds)

    return all_places


def get_reviews_for_area(area: str, cuisine: str = "", max_budget: float = None, max_pages: int = 1) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Fetches restaurant reviews for a specific area in Dubai, dynamically applying cuisine filters,
    calculating empirical price quartiles for missing prices, and filtering by max_budget.

    Returns:
        df_reviews (pd.DataFrame): Extracted review records for NLP analysis.
        df_summary (pd.DataFrame): Data transparency metadata and operational counters.
    """
    if not area or not area.strip():
        raise ValueError("area is required (e.g. 'Dubai Marina').")

    # Build query dynamically based on user inputs
    query_string = f"{cuisine} restaurants in {area}, Dubai".strip()
    places = _fetch_places_pages(query_string, max_pages=max_pages)

    if not places:
        empty_summary = pd.DataFrame([{
            "total_restaurants": 0,
            "total_reviews": 0,
            "exact_price_count": 0,
            "estimated_price_count": 0,
            "transparency_note": f"No restaurants found for '{query_string}'. Try a different area, cuisine, or budget.",
        }])
        return pd.DataFrame(), empty_summary

    # 1. Extract explicit starting prices ONLY if the currency is explicitly AED
    explicit_prices = [
        float(p["priceRange"]["startPrice"]["units"])
        for p in places
        if p.get("priceRange")
        and p["priceRange"].get("startPrice", {}).get("units")
        and p["priceRange"].get("startPrice", {}).get("currencyCode") == "AED" # <-- THE SAFETY CHECK
    ]

    # 2. Compute empirical quartile boundaries for the area
    if len(explicit_prices) >= 4:
        q1, q2, q3 = np.percentile(explicit_prices, [25, 50, 75])
        min_p = min(explicit_prices)
    else:
        # Baseline fallback if explicit price sample size in payload is too small
        min_p, q1, q2, q3 = 25.0, 60.0, 150.0, 350.0

    # 3. Map Google's 4 price categories to the calculated quartiles
    quartile_map = {
        "PRICE_LEVEL_INEXPENSIVE": {"min": min_p, "label": f"AED {int(min_p)} - {int(q1)}"},
        "PRICE_LEVEL_MODERATE": {"min": q1, "label": f"AED {int(q1)} - {int(q2)}"},
        "PRICE_LEVEL_EXPENSIVE": {"min": q2, "label": f"AED {int(q2)} - {int(q3)}"},
        "PRICE_LEVEL_VERY_EXPENSIVE": {"min": q3, "label": f"AED {int(q3)}+"}
    }
    # Places with no price signal at all (no priceRange AND no priceLevel) default
    # to the area's median rather than AED 0, so a strict max_budget filter can't
    # be silently bypassed by unknown-price venues.
    UNKNOWN_PRICE = {"min": q2, "label": "N/A"}

    parsed_reviews = []
    exact_price_count = 0
    estimated_price_count = 0
    analyzed_restaurants = set()

    for place in places:
        place_id = place.get("id")
        restaurant_name = place.get("displayName", {}).get("text")
        cuisine_type = place.get("primaryTypeDisplayName", {}).get("text", "General Dining")

        if not place_id or not restaurant_name:
            continue

        # Extract location coordinates & overall rating counts
        location = place.get("location", {})
        lat = location.get("latitude")
        lng = location.get("longitude")
        user_rating_count = place.get("userRatingCount", 0)

        # Extract actual address
        address = place.get("formattedAddress") or place.get("shortFormattedAddress", "Dubai, UAE")

        # Determine Price & Price Source
        p_range = place.get("priceRange")

        # Check if startPrice exists, has units, AND is strictly in AED
        if (p_range
            and p_range.get("startPrice", {}).get("units")
            and p_range.get("startPrice", {}).get("currencyCode") == "AED"):

            start = float(p_range["startPrice"]["units"])
            end = p_range.get("endPrice", {}).get("units", "")
            min_price = start
            price_display = f"AED {int(start)} - {int(end)}" if end else f"AED {int(start)}+"
            price_source = "Verified Google Price"
            is_exact = True

        else:
            # Fallback for missing prices or non-AED currencies (USD, EUR, etc.)
            raw_level = place.get("priceLevel")
            meta = quartile_map.get(raw_level, UNKNOWN_PRICE)
            min_price = meta["min"]
            price_display = meta["label"]
            price_source = "Area Quartile Estimate" if raw_level in quartile_map else "Unknown (Area Median Used)"
            is_exact = False

        # Filter out venues exceeding user's maximum budget
        if max_budget is not None and min_price > max_budget:
            continue

        # Track metadata counters (keyed by place_id: two branches of the same
        # chain can share a display name, so the name alone isn't a safe key)
        analyzed_restaurants.add(place_id)
        if is_exact:
            exact_price_count += 1
        else:
            estimated_price_count += 1

        # Extract Reviews with timestamps
        for review in place.get("reviews", []):
            text = review.get("text", {}).get("text")
            publish_time = review.get("publishTime", "")

            if text:
                parsed_reviews.append({
                    "place_id": place_id,             # Stable grouping key for all downstream steps
                    "restaurant_name": restaurant_name,
                    "cuisine": cuisine_type,
                    "price_range": price_display,
                    "price_numeric": min_price,        # Numeric price carried through as-is (never re-parsed from the display string)
                    "price_source": price_source,
                    "review_rating": review.get("rating"),
                    "review_text": text,
                    "publish_time": publish_time,           # For Recent Top 3 reviews & date filter
                    "user_rating_count": user_rating_count, # For total review volume column
                    "latitude": lat,                        # For spatial heatmap mapping
                    "longitude": lng,                        # For spatial heatmap mapping
                    "formatted_address": address
                })

    # Construct detailed reviews DataFrame
    df_reviews = pd.DataFrame(parsed_reviews)

    # Construct transparency summary DataFrame
    df_summary = pd.DataFrame([{
        "total_restaurants": len(analyzed_restaurants),
        "total_reviews": len(df_reviews),
        "exact_price_count": exact_price_count,
        "estimated_price_count": estimated_price_count,
        "transparency_note": f"Analyzed {len(analyzed_restaurants)} venues across {len(df_reviews)} reviews. "
                             f"{exact_price_count} using direct menu prices, "
                             f"{estimated_price_count} estimated via local area quartiles."
    }])

    return df_reviews, df_summary


# ==========================================
# EXECUTION & TEST BLOCK
# ==========================================

# Define search parameters
target_area = "Dubai Marina"
target_cuisine = "Italian"
target_max_budget = 150.0

# Run function & unpack both DataFrames
df_reviews, df_summary = get_reviews_for_area(
    area=target_area,
    cuisine=target_cuisine,
    max_budget=target_max_budget
)

# Display transparency banner
print("--- TRANSPARENCY SUMMARY ---")
print(df_summary.loc[0, "transparency_note"])

# Preview reviews DataFrame
print("\n--- REVIEWS DATAFRAME PREVIEW ---")
df_reviews.head(20)


In [ ]:
import numpy as np
import pandas as pd

def compute_advanced_metrics(df_reviews: pd.DataFrame, df_aspects: pd.DataFrame = None) -> pd.DataFrame:
    """
    Computes all advanced metrics and the final composite score for each restaurant.
    Returns a venue-level DataFrame sorted by highest model_score.

    Restaurants are grouped by `place_id` (not `restaurant_name`) because two
    branches of the same chain within one area can share a display name.

    df_aspects (optional): output of classify_review_aspects() -- one row per
    (place_id, aspect, is_positive) sentence-level hit, produced by zero-shot
    classification instead of fixed keyword lists. When omitted (e.g. the raw,
    pre-sentiment df_reviews), aspect/value scores fall back to neutral baselines.
    """
    if df_reviews.empty:
        return pd.DataFrame()

    # 1. Take top 5 latest reviews per restaurant
    df_reviews = df_reviews.copy()
    df_reviews["publish_time"] = pd.to_datetime(df_reviews["publish_time"], errors="coerce")
    top5_reviews = (
        df_reviews.sort_values(by=["place_id", "publish_time"], ascending=[True, False])
        .groupby("place_id")
        .head(5)
    )

    # 2. Extract static venue metadata (1 row per restaurant)
    venues = top5_reviews.groupby("place_id", as_index=False).first()
    venues = venues.rename(columns={"user_rating_count": "total_google_ratings"})

    # 3. Recalculate true averages across those 5 reviews
    venues["avg_google_rating"] = venues["place_id"].map(
        top5_reviews.groupby("place_id")["review_rating"].mean()
    ).round(1)

    if "sentiment_label" in top5_reviews.columns:
        venues["positive_sentiment_pct"] = venues["place_id"].map(
            top5_reviews.groupby("place_id")["sentiment_label"].apply(
                lambda s: (s == "POSITIVE").mean() * 100.0
            )
        ).round(1)
    else:
        venues["positive_sentiment_pct"] = venues["place_id"].map(
            top5_reviews.groupby("place_id")["review_rating"].apply(
                lambda r: (r >= 4).mean() * 100.0
            )
        ).round(1)

    # Clean display fields. price_numeric is carried straight through from the
    # API fetch step, never re-derived from the "AED X - Y" display string
    # (that string has no "-" for the top price tier or open-ended prices,
    # which used to silently collapse those venues to a flat fallback price).
    venues["short_formatted_address"] = venues["formatted_address"].apply(
        lambda addr: str(addr).split("-")[0].strip()
    )

    # -------------------------------------------------------------------------
    # CALCULATE THE 5 METRICS
    # -------------------------------------------------------------------------
    # 1. Volume Confidence
    venues["volume_confidence_score"] = venues["total_google_ratings"].apply(
        lambda x: min(100.0, (np.log10(x + 1) / np.log10(2500.0)) * 100.0)
    )

    # 2. Sentiment Momentum
    venues["avg_google_rating_scaled"] = (venues["avg_google_rating"] / 5.0) * 100.0
    venues["sentiment_momentum_score"] = np.clip(
        50.0 + (venues["positive_sentiment_pct"] - venues["avg_google_rating_scaled"]),
        0.0, 100.0
    )

    # 3. Aspect Sentiment (Food & Service)
    # Driven by zero-shot aspect classification (see classify_review_aspects in
    # the NLP cell) instead of a fixed keyword list, so paraphrases a keyword
    # list would miss (e.g. "a bit steep for what you get") are still caught.
    # Venues with no food/service-classified sentences fall back to their
    # overall positive_sentiment_pct.
    if df_aspects is not None and not df_aspects.empty:
        fs_hits = df_aspects[df_aspects["aspect"].isin(["food quality", "service quality"])]
        aspect_scores = (fs_hits.groupby("place_id")["is_positive"].mean() * 100.0).to_dict()
    else:
        aspect_scores = {}

    venues["aspect_sentiment_score"] = venues["place_id"].map(aspect_scores)
    venues["aspect_sentiment_score"] = venues["aspect_sentiment_score"].fillna(venues["positive_sentiment_pct"])

    # 4. Negativity Risk Penalty
    risk_scores = {}
    for place_id, group in top5_reviews.groupby("place_id"):
        high_risk = group.apply(
            lambda r: (r.get("review_rating", 5) <= 2) or (r.get("sentiment_label") == "NEGATIVE" and r.get("sentiment_score", 0.0) >= 0.85),
            axis=1
        )
        risk_scores[place_id] = float((high_risk.mean()) * 100.0)

    venues["negativity_risk_score"] = venues["place_id"].map(risk_scores).fillna(0.0)

    # 5. Price-to-Value Index
    # Driven by the same zero-shot classification: sentences classified as
    # "price or value" contribute their sentence-level sentiment polarity,
    # replacing the fixed pos/neg value-phrase keyword lists. Venues with no
    # price/value-classified sentences fall back to a neutral 50.0.
    if df_aspects is not None and not df_aspects.empty:
        value_hits = df_aspects[df_aspects["aspect"] == "price or value"]
        value_scores = (value_hits.groupby("place_id")["is_positive"].mean() * 100.0).to_dict()
    else:
        value_scores = {}

    venues["value_keyword_score"] = venues["place_id"].map(value_scores).fillna(50.0)
    venues["price_factor"] = venues["price_numeric"].apply(lambda p: max(50.0, 100.0 - ((p / 500.0) * 50.0)))
    venues["price_value_score"] = (
        (0.50 * venues["positive_sentiment_pct"]) +
        (0.30 * venues["value_keyword_score"]) +
        (0.20 * venues["price_factor"])
    )

    # -------------------------------------------------------------------------
    # COMPOSITE SCORE
    # -------------------------------------------------------------------------
    raw_model_score = (
        (0.30 * venues["positive_sentiment_pct"]) +
        (0.25 * venues["avg_google_rating_scaled"]) +
        (0.15 * venues["volume_confidence_score"]) +
        (0.10 * venues["sentiment_momentum_score"]) +
        (0.10 * venues["aspect_sentiment_score"]) +
        (0.10 * venues["price_value_score"]) -
        (0.10 * venues["negativity_risk_score"])
    )

    tie_breaker = (venues["total_google_ratings"] % 97) * 0.001
    venues["model_score"] = np.clip(raw_model_score + tie_breaker, 0.0, 100.0).round(2)

    # Return the fully scored dataframe sorted by rank
    df_scored = venues.sort_values(by="model_score", ascending=False).reset_index(drop=True)
    return df_scored


In [ ]:
df_venues = compute_advanced_metrics(df_reviews)

df_reviews_with_scores = df_reviews.merge(
    df_venues[["place_id", "model_score"]],
    on="place_id",
    how="left"
)
df_reviews_with_scores.head()


**Run Hugging Face Sentiment Analysis Model**

In [ ]:
df_analyzed = analyze_sentiment(df_reviews)
df_analyzed.head()


In [ ]:
import os
import nltk
from collections import Counter
import pandas as pd
import torch
import spacy
from spacy.cli import download
from transformers import pipeline


def _get_hf_token():
    """Fetches the Hugging Face Hub token from Colab Secrets when available,
    falling back to the HF_TOKEN environment variable everywhere else. Not
    required for these public models to load, but avoids the "unauthenticated
    requests" rate-limit warning and speeds up downloads."""
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")  # OK to stay None -- these are public models


HF_TOKEN = _get_hf_token()

# --- 1. SAFE SPACY MODEL LOADER ---
def load_spacy_model():
    """Safely loads spaCy model; downloads automatically if missing locally or in Colab."""
    try:
        return spacy.load("en_core_web_sm")
    except OSError:
        download("en_core_web_sm")
        return spacy.load("en_core_web_sm")

nlp = load_spacy_model()

# --- 2. GLOBAL TRANSFORMER PIPELINES ---
# Auto-detects a GPU when available (e.g. Colab) and falls back to CPU
# (e.g. a plain Docker container) instead of crashing on CPU-only hosts.
_device = 0 if torch.cuda.is_available() else -1
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=_device,
    token=HF_TOKEN
)

# Zero-shot classifier for aspect detection (food/service/price/ambiance).
# Replaces fixed keyword lists (food_kw, service_kw, pos/neg value terms):
# a keyword list only ever matches the exact words someone typed it with,
# while this generalizes to paraphrases (e.g. "a bit steep for what you
# get" reads as negative "price or value" even though no keyword matches).
# distilbart-mnli-12-3 is a distilled, CPU-friendly zero-shot model -- a
# meaningfully larger download than a keyword list, so its cost is paid
# once at process startup, not per request.
ASPECT_LABELS = ["food quality", "service quality", "price or value", "ambiance"]
ASPECT_CONFIDENCE_THRESHOLD = 0.5

aspect_classifier = pipeline(
    "zero-shot-classification",
    model="valhalla/distilbart-mnli-12-3",
    device=_device,
    token=HF_TOKEN
)

# --- 3. DYNAMIC DISH & VIBE EXTRACTION ---
# Famous-dish extraction is structural (dependency parsing + NER) instead of
# a single "not a dish" blocklist checked against every noun chunk. That
# blocklist approach let anything it didn't anticipate leak through as a
# "dish" -- e.g. "I love Dubai" produced "Dubai" (a place name, picked up
# because "love" is a generic verb, not a food-specific one), and "the
# attention to detail was great" produced "Detail" (a generic noun the list
# never covered). Two-tier fix:
#  1. Primary signal: the dish is the direct object of a genuinely
#     food-specific verb (order/try/recommend/taste). High precision, no
#     word list needed for this path at all.
#  2. Fallback signal: the dish is the subject of "X was <adjective>" --
#     structurally identical for food nouns and generic nouns alike, so it
#     still needs a residual guard list; kept small but complete enough to
#     actually work.
# Both paths exclude any candidate overlapping a real-world place/
# organization/person/nationality entity, using spaCy's own NER (already
# loaded) instead of hardcoding place names like "Dubai" one by one.
# Consumption-verb set is computed from WordNet (nltk), not hand-typed:
# each seed word is expanded to the synonym set for one specific sense --
#  - order      (sense: "make a request for something" -- ordering at a restaurant)
#  - taste      (sense: "take a sample of" -- also yields try/sample as synonyms)
#  - recommend  (sense: "push for something" -- also yields urge/advocate)
# Falls back to just the seed words if nltk/WordNet is unavailable (e.g. no
# network access to fetch the corpus on first use).
def _wordnet_verb_synonyms(seed_verb: str, sense_index: int) -> set:
    try:
        from nltk.corpus import wordnet as wn
        try:
            synsets = wn.synsets(seed_verb, pos=wn.VERB)
        except LookupError:
            nltk.download("wordnet", quiet=True)
            synsets = wn.synsets(seed_verb, pos=wn.VERB)
        if synsets and sense_index < len(synsets):
            return {lemma.split("_")[0].lower() for lemma in synsets[sense_index].lemma_names()}
    except Exception:
        pass
    return {seed_verb}


CONSUMPTION_VERBS = (
    _wordnet_verb_synonyms("order", 1)
    | _wordnet_verb_synonyms("taste", 2)
    | _wordnet_verb_synonyms("recommend", 0)
)

NON_FOOD_ENTITY_LABELS = {"GPE", "LOC", "ORG", "PERSON", "NORP", "FAC"}

GENERIC_FALLBACK_WORDS = {
    "place", "restaurant", "experience", "food", "menu", "price", "view",
    "service", "staff", "detail", "attention", "presentation", "wait",
    "crowd", "night", "dinner", "lunch", "visit", "quality", "thing",
    "ambiance", "atmosphere", "value", "way", "time", "portion", "spot"
}


def _entity_token_spans(doc):
    """Token indices covered by non-food named entities (places, orgs,
    people, nationalities), so dish candidates can avoid them."""
    spans = set()
    for ent in doc.ents:
        if ent.label_ in NON_FOOD_ENTITY_LABELS:
            spans.update(range(ent.start, ent.end))
    return spans


def _phrase_for_token(token, doc, entity_spans):
    """Expands a single token to its enclosing noun chunk (e.g. 'ouzi' -> 'Lamb Ouzi'),
    skipping any chunk that overlaps a non-food named entity."""
    for chunk in doc.noun_chunks:
        if chunk.start <= token.i < chunk.end:
            if entity_spans & set(range(chunk.start, chunk.end)):
                return None
            words = [
                t.text.lower() for t in chunk
                if t.pos_ in ("NOUN", "PROPN") and not t.is_stop and t.is_alpha
            ]
            if words and len(words) <= 3:
                return " ".join(words).title()
    return None


DEFAULT_VIBE_FREQUENCIES = {
    "Welcoming": 1, "Cozy": 1, "Lively": 1, "Warm": 1,
    "Friendly": 1, "Excellent": 1, "Great": 1, "Elegant": 1
}

def extract_dish_and_vibe(texts: list) -> tuple[str, str, dict]:
    """
    Dynamically extracts dish candidates and vibe keywords using spaCy:
    - Vibes: adjective frequency count for the WordCloud, plus a short
      comma-joined top-10 string for any plain-text display.
    - Famous Dish: verb-object + NER structural extraction (see comment above).
    """
    if not texts:
        return "Chef Special", "Welcoming, Cozy, Lively, Warm, Friendly, Excellent, Great, Elegant", dict(DEFAULT_VIBE_FREQUENCIES)

    doc = nlp(" ".join(texts))

    # 1. Vibe Check: full adjective frequency count (word -> count) for the
    # WordCloud -- most wordcloud libraries (e.g. Python's `wordcloud`
    # package) take a frequency dict directly, so a flat top-10 list alone
    # throws away the weighting a wordcloud needs and looks sparse.
    vibe_counts = Counter(
        token.lemma_.title() for token in doc
        if token.pos_ == "ADJ"
        and not token.is_stop
        and token.is_alpha
        and len(token.text) > 2
    )

    top_vibes = [v[0] for v in vibe_counts.most_common(10)]
    vibe_check = ", ".join(top_vibes) if top_vibes else "Welcoming, Cozy, Lively, Warm, Friendly, Excellent, Great, Elegant"
    vibe_word_frequencies = dict(vibe_counts.most_common(50)) if vibe_counts else dict(DEFAULT_VIBE_FREQUENCIES)

    # 2. Famous Dish
    entity_spans = _entity_token_spans(doc)
    primary_candidates = []   # dish is the object of a food-specific verb
    fallback_candidates = []  # dish is the subject of "X was <adjective>"

    for token in doc:
        if token.dep_ in ("dobj", "obj") and token.head.lemma_.lower() in CONSUMPTION_VERBS:
            phrase = _phrase_for_token(token, doc, entity_spans)
            if phrase:
                primary_candidates.append(phrase)

        elif token.dep_ == "nsubj" and token.head.lemma_.lower() == "be":
            phrase = _phrase_for_token(token, doc, entity_spans)
            if phrase and not set(phrase.lower().split()) & GENERIC_FALLBACK_WORDS:
                fallback_candidates.append(phrase)

    dish_candidates = primary_candidates or fallback_candidates
    top_dishes = Counter(dish_candidates).most_common(1)
    famous_dish = top_dishes[0][0] if top_dishes else "Chef Special"

    return famous_dish, vibe_check, vibe_word_frequencies

# --- 4. SENTIMENT INFERENCE ENGINE ---
def analyze_sentiment(df_reviews: pd.DataFrame) -> pd.DataFrame:
    """
    Runs batched Hugging Face transformer inference while preserving all
    metadata columns (review_text, latitude, longitude, publish_time, user_rating_count).
    """
    if df_reviews.empty or 'review_text' not in df_reviews.columns:
        return df_reviews

    texts = df_reviews['review_text'].tolist()
    predictions = sentiment_analyzer(texts, truncation=True, max_length=512)

    df_reviews['sentiment_label'] = [pred['label'] for pred in predictions]
    df_reviews['sentiment_score'] = [round(pred['score'], 4) for pred in predictions]

    return df_reviews

# --- 5. ASPECT CLASSIFICATION (replaces fixed keyword lists) ---
def classify_review_aspects(df_reviews: pd.DataFrame) -> pd.DataFrame:
    """
    Splits each review into sentences (spaCy) and, per sentence, uses
    zero-shot classification to detect which aspect(s) it discusses
    (food/service/price/ambiance) plus the existing sentiment model for
    that sentence's polarity. Returns one row per (place_id, aspect,
    is_positive) sentence-aspect hit above ASPECT_CONFIDENCE_THRESHOLD,
    consumed by compute_advanced_metrics for aspect_sentiment_score and
    value_keyword_score.
    """
    if df_reviews.empty or "review_text" not in df_reviews.columns:
        return pd.DataFrame(columns=["place_id", "aspect", "is_positive"])

    sentences, sentence_place_ids = [], []
    for _, row in df_reviews.iterrows():
        text = str(row.get("review_text", ""))
        if not text.strip():
            continue
        for sent in nlp(text).sents:
            clean = sent.text.strip()
            if len(clean) >= 3:  # drop stray punctuation/empty fragments from sentence splitting
                sentences.append(clean)
                sentence_place_ids.append(row["place_id"])

    if not sentences:
        return pd.DataFrame(columns=["place_id", "aspect", "is_positive"])

    aspect_predictions = aspect_classifier(sentences, ASPECT_LABELS, multi_label=True)
    sentiment_predictions = sentiment_analyzer(sentences, truncation=True, max_length=512)

    rows = []
    for place_id, aspect_pred, sentiment_pred in zip(sentence_place_ids, aspect_predictions, sentiment_predictions):
        is_positive = sentiment_pred["label"] == "POSITIVE"
        for label, score in zip(aspect_pred["labels"], aspect_pred["scores"]):
            if score >= ASPECT_CONFIDENCE_THRESHOLD:
                rows.append({"place_id": place_id, "aspect": label, "is_positive": is_positive})

    return pd.DataFrame(rows)


# --- 6. DASHBOARD INSIGHTS AGGREGATOR ---
def generate_restaurant_insights(df_analyzed: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Aggregates sentiment, extracts dynamic dish/vibe tags, preserves spatial & volume
    metadata for dashboard charts, and ranks the #1 'Rising Star' venue based on model_score.

    Grouped by `place_id` (not `restaurant_name`) for the same reason as
    compute_advanced_metrics: display names aren't unique across branches.
    """
    if df_analyzed.empty:
        return pd.DataFrame(), {}

    # Cast timestamps to datetime
    if "publish_time" in df_analyzed.columns:
        df_analyzed["publish_time"] = pd.to_datetime(df_analyzed["publish_time"], errors="coerce")

    agg_list = []
    for place_id, group in df_analyzed.groupby("place_id"):
        total_reviews = len(group)
        pos_reviews = (group["sentiment_label"] == "POSITIVE").sum()
        pos_ratio = round((pos_reviews / total_reviews) * 100, 1)
        avg_rating = round(group["review_rating"].mean(), 1)

        # Preserved metadata fields & PRE-CALCULATED MODEL SCORE
        restaurant_name = group["restaurant_name"].iloc[0]
        price_range = group["price_range"].iloc[0]
        cuisine = group["cuisine"].iloc[0]
        lat = group["latitude"].iloc[0] if "latitude" in group.columns else None
        lng = group["longitude"].iloc[0] if "longitude" in group.columns else None
        user_rating_count = group["user_rating_count"].iloc[0] if "user_rating_count" in group.columns else 0
        price_numeric = group["price_numeric"].iloc[0] if "price_numeric" in group.columns else None

        # Pulling the model score calculated upstream
        model_score = group["model_score"].iloc[0] if "model_score" in group.columns else 0.0

        # Dynamic NLP dish/vibe tags
        pos_texts = group[group["sentiment_label"] == "POSITIVE"]["review_text"].tolist()
        famous_dish, vibe_check, vibe_word_frequencies = extract_dish_and_vibe(pos_texts if pos_texts else group["review_text"].tolist())

        # Sort chronologically for Recent Top 3 Review card
        if "publish_time" in group.columns and group["publish_time"].notna().any():
            recent_group = group.sort_values(by="publish_time", ascending=False)
        else:
            recent_group = group

        recent_3_reviews = recent_group[
            ["review_text", "publish_time", "review_rating", "sentiment_label"]
        ].head(3).to_dict(orient="records")

        # Top sample quotes
        pos_snippet = group[group["sentiment_label"] == "POSITIVE"]["review_text"].head(1).values
        neg_snippet = group[group["sentiment_label"] == "NEGATIVE"]["review_text"].head(1).values

        agg_list.append({
            "place_id": place_id,
            "restaurant_name": restaurant_name,
            "model_score": model_score,  # Passed straight through
            "cuisine": cuisine,
            "price_range": price_range,
            "price_numeric": price_numeric,
            "avg_google_rating": avg_rating,
            "total_google_ratings": user_rating_count,
            "positive_sentiment_pct": pos_ratio,
            "reviews_analyzed": total_reviews,
            "famous_dish": famous_dish,
            "vibe_check": vibe_check,
            "vibe_word_frequencies": vibe_word_frequencies,
            "latitude": lat,
            "longitude": lng,
            "recent_3_reviews": recent_3_reviews,
            "sample_positive_review": pos_snippet[0] if len(pos_snippet) > 0 else "N/A",
            "sample_negative_review": neg_snippet[0] if len(neg_snippet) > 0 else "N/A"
        })

    df_scorecard = pd.DataFrame(agg_list)

    # Rank venues strictly by the pre-computed model_score
    df_scorecard = df_scorecard.sort_values(
        by=["model_score", "positive_sentiment_pct"],
        ascending=[False, False]
    ).reset_index(drop=True)

    top_venue = df_scorecard.iloc[0].to_dict() if not df_scorecard.empty else {}

    return df_scorecard, top_venue


In [ ]:
# ==========================================
# END-TO-END EXECUTION & OUTPUT TEST
# ==========================================

# 1. COMPUTE SCORES: Run aspect classification, then the advanced model math
df_aspects = classify_review_aspects(df_analyzed)
df_scores = compute_advanced_metrics(df_analyzed, df_aspects)

# Prevent "model_score_x" / "model_score_y" duplication if you run this cell multiple times
if "model_score" in df_analyzed.columns:
    df_analyzed = df_analyzed.drop(columns=["model_score"])

# Attach the new scores to the review-level data (joined on place_id, not name)
df_analyzed = df_analyzed.merge(
    df_scores[["place_id", "model_score"]],
    on="place_id",
    how="left"
)

# 2. Aggregate insights, extract NLP tags, and build the final scorecard
df_scorecard, top_venue = generate_restaurant_insights(df_analyzed)

# --- PRINT OUTPUTS FOR VERIFICATION ---

print("=== ℹ️ TRANSPARENCY BANNER ===")
try:
    print(df_summary.loc[0, "transparency_note"])
except NameError:
    print("Data loaded from local memory (API filters pre-applied).")

print("\n=== 🏆 RISING STAR VENUE (Ranked by Advanced Model) ===")
print(f"Name:                   {top_venue.get('restaurant_name')}")
print(f"COMPOSITE MODEL SCORE:  {top_venue.get('model_score')} / 100")
print(f"Cuisine:                {top_venue.get('cuisine')}")
print(f"Price Range:            {top_venue.get('price_range')}")
print(f"Positive Sentiment:     {top_venue.get('positive_sentiment_pct')}%")
print(f"Avg Google Rating:      {top_venue.get('avg_google_rating')} ⭐ ({top_venue.get('total_google_ratings')} ratings)")
print(f"Famous For Dish:        {top_venue.get('famous_dish')}")
print(f"Vibe Check:             {top_venue.get('vibe_check')}")
print(f"Coordinates (Lat, Lng): ({top_venue.get('latitude')}, {top_venue.get('longitude')})")

print("\n--- RECENT TOP 3 REVIEWS ---")
for idx, rev in enumerate(top_venue.get("recent_3_reviews", []), 1):
    # Strip line breaks and strictly truncate to ~130 characters (max 2 lines)
    clean_text = str(rev.get('review_text', '')).replace('\n', ' ').strip()
    if len(clean_text) > 130:
        clean_text = clean_text[:127] + "..."

    print(f"{idx}. [{rev['sentiment_label']}] Rating: {rev['review_rating']}⭐ - \"{clean_text}\"")

print("\n=== 📊 FULL RESTAURANT SCORECARD TABLE ===")
display_columns = [
    "restaurant_name",
    "model_score",
    "positive_sentiment_pct",
    "avg_google_rating",
    "price_range",
    "total_google_ratings",
    "famous_dish"
]

# Leaving this without print() so Jupyter renders a clean HTML table
df_scorecard[display_columns].head(10)


In [ ]:
# ==========================================
# LIVE RECOMMENDATION ORCHESTRATOR + CACHE
# ==========================================
# Wraps the full pipeline (fetch -> sentiment -> scoring -> insights) behind a
# single function the API layer can call per request, with a TTL cache so
# repeat (area, cuisine, budget) queries don't re-hit the Places API or re-run
# NLP inference every time. Swappable for Redis later without touching callers.
import time


class TTLCache:
    def __init__(self, ttl_seconds: float = 6 * 3600):
        self.ttl_seconds = ttl_seconds
        self._store = {}

    def get(self, key):
        entry = self._store.get(key)
        if entry is None:
            return None
        value, expires_at = entry
        if time.time() > expires_at:
            del self._store[key]
            return None
        return value

    def set(self, key, value):
        self._store[key] = (value, time.time() + self.ttl_seconds)


recommendation_cache = TTLCache(ttl_seconds=6 * 3600)  # reviews/prices don't change minute-to-minute


def _jsonable(obj):
    """Recursively converts numpy/pandas scalar types into plain JSON-safe values."""
    if isinstance(obj, dict):
        return {k: _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        val = float(obj)
        return None if np.isnan(val) else val
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat() if pd.notnull(obj) else None
    if isinstance(obj, float) and pd.isna(obj):
        return None
    return obj


def _aggregate_stats(df_scorecard: pd.DataFrame) -> dict:
    """Cross-restaurant summary stats for the dashboard's top scorecards."""
    return {
        "restaurants_analyzed": int(len(df_scorecard)),
        "reviews_analyzed": int(df_scorecard["reviews_analyzed"].sum()),
        "avg_google_rating": round(float(df_scorecard["avg_google_rating"].mean()), 1),
        "avg_sentiment_pct": round(float(df_scorecard["positive_sentiment_pct"].mean()), 1),
        "avg_spend_per_person": round(float(df_scorecard["price_numeric"].dropna().mean()), 0)
            if df_scorecard["price_numeric"].notna().any() else None,
    }


def get_recommendations(area: str, cuisine: str = "", max_budget: float = None, use_cache: bool = True) -> dict:
    """
    End-to-end pipeline for one user request.
    Returns a JSON-serializable dict: {"summary": str, "top_pick": dict|None,
    "restaurants": [dict, ...], "stats": dict} -- "stats" holds the
    cross-restaurant aggregates (restaurants/reviews analyzed, avg rating,
    avg sentiment, avg spend per person) for the dashboard's scorecards.
    """
    cache_key = (area.strip().lower(), (cuisine or "").strip().lower(), max_budget)

    if use_cache:
        cached = recommendation_cache.get(cache_key)
        if cached is not None:
            return cached

    df_reviews, df_summary = get_reviews_for_area(area=area, cuisine=cuisine, max_budget=max_budget)

    if df_reviews.empty:
        result = {
            "summary": df_summary.loc[0, "transparency_note"],
            "top_pick": None,
            "restaurants": [],
            "stats": {
                "restaurants_analyzed": 0, "reviews_analyzed": 0,
                "avg_google_rating": None, "avg_sentiment_pct": None, "avg_spend_per_person": None,
            },
        }
        if use_cache:
            recommendation_cache.set(cache_key, result)
        return result

    df_analyzed = analyze_sentiment(df_reviews)
    df_aspects = classify_review_aspects(df_analyzed)
    df_scores = compute_advanced_metrics(df_analyzed, df_aspects)
    df_analyzed = df_analyzed.merge(df_scores[["place_id", "model_score"]], on="place_id", how="left")
    df_scorecard, top_venue = generate_restaurant_insights(df_analyzed)

    result = _jsonable({
        "summary": df_summary.loc[0, "transparency_note"],
        "top_pick": top_venue if top_venue else None,
        "restaurants": df_scorecard.to_dict(orient="records"),
        "stats": _aggregate_stats(df_scorecard),
    })

    if use_cache:
        recommendation_cache.set(cache_key, result)

    return result


# Quick smoke test against the pipeline you already validated above
sample_result = get_recommendations(area="Dubai Marina", cuisine="Italian", max_budget=150.0)
print(sample_result["summary"])
print(f"Top pick: {sample_result['top_pick'].get('restaurant_name') if sample_result['top_pick'] else 'None'}")


In [ ]:
%%writefile server.py
import os
import time
import nltk
from collections import Counter

import numpy as np
import pandas as pd
import requests
import torch
import spacy
from spacy.cli import download
from transformers import pipeline
from fastapi import FastAPI, HTTPException, Query
from fastapi.responses import FileResponse


def _get_hf_token():
    """Fetches the Hugging Face Hub token from Colab Secrets when available,
    falling back to the HF_TOKEN environment variable everywhere else. Not
    required for these public models to load, but avoids the "unauthenticated
    requests" rate-limit warning and speeds up downloads."""
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")  # OK to stay None -- these are public models


HF_TOKEN = _get_hf_token()
from fastapi.middleware.cors import CORSMiddleware

# ==========================================
# CONFIG & MODEL LOADING (once, at process startup)
# ==========================================
API_KEY = os.environ.get("GOOGLE_PLACES_API_KEY")
if not API_KEY:
    raise RuntimeError("GOOGLE_PLACES_API_KEY environment variable is not set.")

PLACES_SEARCH_URL = "https://places.googleapis.com/v1/places:searchText"
FIELD_MASK = (
    "places.id,"
    "places.displayName,"
    "places.rating,"
    "places.userRatingCount,"
    "places.location,"
    "places.reviews,"
    "places.priceLevel,"
    "places.priceRange,"
    "places.primaryTypeDisplayName,"
    "places.formattedAddress,"
    "places.shortFormattedAddress,"
    "nextPageToken"
)


def load_spacy_model():
    try:
        return spacy.load("en_core_web_sm")
    except OSError:
        download("en_core_web_sm")
        return spacy.load("en_core_web_sm")


nlp = load_spacy_model()

# Auto-detects GPU when available, falls back to CPU (this is a CPU deployment target).
_device = 0 if torch.cuda.is_available() else -1
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=_device,
    token=HF_TOKEN,
)

# Zero-shot aspect classifier (food/service/price/ambiance), replacing fixed
# keyword lists so paraphrases are still picked up. Loaded once at startup.
ASPECT_LABELS = ["food quality", "service quality", "price or value", "ambiance"]
ASPECT_CONFIDENCE_THRESHOLD = 0.5
aspect_classifier = pipeline(
    "zero-shot-classification",
    model="valhalla/distilbart-mnli-12-3",
    device=_device,
    token=HF_TOKEN,
)


# ==========================================
# CORE PIPELINE (fetch -> sentiment -> scoring -> insights)
# Kept in lockstep with the notebook cells this logic was validated in.
# ==========================================
def _fetch_places_pages(query_string, max_pages=1, page_delay_seconds=2.0):
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": FIELD_MASK,
    }
    all_places = []
    page_token = None
    for _ in range(max_pages):
        payload = {"textQuery": query_string, "languageCode": "en", "regionCode": "AE"}
        if page_token:
            payload["pageToken"] = page_token
        try:
            response = requests.post(PLACES_SEARCH_URL, headers=headers, json=payload, timeout=15)
            response.raise_for_status()
        except requests.exceptions.RequestException as exc:
            raise RuntimeError(f"Google Places API request failed: {exc}") from exc
        data = response.json()
        all_places.extend(data.get("places", []))
        page_token = data.get("nextPageToken")
        if not page_token:
            break
        time.sleep(page_delay_seconds)
    return all_places


def get_reviews_for_area(area, cuisine="", max_budget=None, max_pages=1):
    if not area or not area.strip():
        raise ValueError("area is required (e.g. 'Dubai Marina').")

    query_string = f"{cuisine} restaurants in {area}, Dubai".strip()
    places = _fetch_places_pages(query_string, max_pages=max_pages)

    if not places:
        empty_summary = pd.DataFrame([{
            "total_restaurants": 0,
            "total_reviews": 0,
            "exact_price_count": 0,
            "estimated_price_count": 0,
            "transparency_note": f"No restaurants found for '{query_string}'. Try a different area, cuisine, or budget.",
        }])
        return pd.DataFrame(), empty_summary

    explicit_prices = [
        float(p["priceRange"]["startPrice"]["units"])
        for p in places
        if p.get("priceRange")
        and p["priceRange"].get("startPrice", {}).get("units")
        and p["priceRange"].get("startPrice", {}).get("currencyCode") == "AED"
    ]

    if len(explicit_prices) >= 4:
        q1, q2, q3 = np.percentile(explicit_prices, [25, 50, 75])
        min_p = min(explicit_prices)
    else:
        min_p, q1, q2, q3 = 25.0, 60.0, 150.0, 350.0

    quartile_map = {
        "PRICE_LEVEL_INEXPENSIVE": {"min": min_p, "label": f"AED {int(min_p)} - {int(q1)}"},
        "PRICE_LEVEL_MODERATE": {"min": q1, "label": f"AED {int(q1)} - {int(q2)}"},
        "PRICE_LEVEL_EXPENSIVE": {"min": q2, "label": f"AED {int(q2)} - {int(q3)}"},
        "PRICE_LEVEL_VERY_EXPENSIVE": {"min": q3, "label": f"AED {int(q3)}+"},
    }
    UNKNOWN_PRICE = {"min": q2, "label": "N/A"}

    parsed_reviews = []
    exact_price_count = 0
    estimated_price_count = 0
    analyzed_restaurants = set()

    for place in places:
        place_id = place.get("id")
        restaurant_name = place.get("displayName", {}).get("text")
        cuisine_type = place.get("primaryTypeDisplayName", {}).get("text", "General Dining")
        if not place_id or not restaurant_name:
            continue

        location = place.get("location", {})
        lat = location.get("latitude")
        lng = location.get("longitude")
        user_rating_count = place.get("userRatingCount", 0)
        address = place.get("formattedAddress") or place.get("shortFormattedAddress", "Dubai, UAE")
        p_range = place.get("priceRange")

        if (p_range
            and p_range.get("startPrice", {}).get("units")
            and p_range.get("startPrice", {}).get("currencyCode") == "AED"):
            start = float(p_range["startPrice"]["units"])
            end = p_range.get("endPrice", {}).get("units", "")
            min_price = start
            price_display = f"AED {int(start)} - {int(end)}" if end else f"AED {int(start)}+"
            price_source = "Verified Google Price"
            is_exact = True
        else:
            raw_level = place.get("priceLevel")
            meta = quartile_map.get(raw_level, UNKNOWN_PRICE)
            min_price = meta["min"]
            price_display = meta["label"]
            price_source = "Area Quartile Estimate" if raw_level in quartile_map else "Unknown (Area Median Used)"
            is_exact = False

        if max_budget is not None and min_price > max_budget:
            continue

        analyzed_restaurants.add(place_id)
        if is_exact:
            exact_price_count += 1
        else:
            estimated_price_count += 1

        for review in place.get("reviews", []):
            text = review.get("text", {}).get("text")
            publish_time = review.get("publishTime", "")
            if text:
                parsed_reviews.append({
                    "place_id": place_id,
                    "restaurant_name": restaurant_name,
                    "cuisine": cuisine_type,
                    "price_range": price_display,
                    "price_numeric": min_price,
                    "price_source": price_source,
                    "review_rating": review.get("rating"),
                    "review_text": text,
                    "publish_time": publish_time,
                    "user_rating_count": user_rating_count,
                    "latitude": lat,
                    "longitude": lng,
                    "formatted_address": address,
                })

    df_reviews = pd.DataFrame(parsed_reviews)
    df_summary = pd.DataFrame([{
        "total_restaurants": len(analyzed_restaurants),
        "total_reviews": len(df_reviews),
        "exact_price_count": exact_price_count,
        "estimated_price_count": estimated_price_count,
        "transparency_note": f"Analyzed {len(analyzed_restaurants)} venues across {len(df_reviews)} reviews. "
                             f"{exact_price_count} using direct menu prices, "
                             f"{estimated_price_count} estimated via local area quartiles.",
    }])
    return df_reviews, df_summary


def analyze_sentiment(df_reviews):
    if df_reviews.empty or "review_text" not in df_reviews.columns:
        return df_reviews
    texts = df_reviews["review_text"].tolist()
    predictions = sentiment_analyzer(texts, truncation=True, max_length=512)
    df_reviews["sentiment_label"] = [pred["label"] for pred in predictions]
    df_reviews["sentiment_score"] = [round(pred["score"], 4) for pred in predictions]
    return df_reviews


def compute_advanced_metrics(df_reviews, df_aspects=None):
    if df_reviews.empty:
        return pd.DataFrame()

    df_reviews = df_reviews.copy()
    df_reviews["publish_time"] = pd.to_datetime(df_reviews["publish_time"], errors="coerce")
    top5_reviews = (
        df_reviews.sort_values(by=["place_id", "publish_time"], ascending=[True, False])
        .groupby("place_id")
        .head(5)
    )

    venues = top5_reviews.groupby("place_id", as_index=False).first()
    venues = venues.rename(columns={"user_rating_count": "total_google_ratings"})

    venues["avg_google_rating"] = venues["place_id"].map(
        top5_reviews.groupby("place_id")["review_rating"].mean()
    ).round(1)

    if "sentiment_label" in top5_reviews.columns:
        venues["positive_sentiment_pct"] = venues["place_id"].map(
            top5_reviews.groupby("place_id")["sentiment_label"].apply(
                lambda s: (s == "POSITIVE").mean() * 100.0
            )
        ).round(1)
    else:
        venues["positive_sentiment_pct"] = venues["place_id"].map(
            top5_reviews.groupby("place_id")["review_rating"].apply(
                lambda r: (r >= 4).mean() * 100.0
            )
        ).round(1)

    venues["short_formatted_address"] = venues["formatted_address"].apply(
        lambda addr: str(addr).split("-")[0].strip()
    )

    venues["volume_confidence_score"] = venues["total_google_ratings"].apply(
        lambda x: min(100.0, (np.log10(x + 1) / np.log10(2500.0)) * 100.0)
    )

    venues["avg_google_rating_scaled"] = (venues["avg_google_rating"] / 5.0) * 100.0
    venues["sentiment_momentum_score"] = np.clip(
        50.0 + (venues["positive_sentiment_pct"] - venues["avg_google_rating_scaled"]),
        0.0, 100.0
    )

    if df_aspects is not None and not df_aspects.empty:
        fs_hits = df_aspects[df_aspects["aspect"].isin(["food quality", "service quality"])]
        aspect_scores = (fs_hits.groupby("place_id")["is_positive"].mean() * 100.0).to_dict()
    else:
        aspect_scores = {}

    venues["aspect_sentiment_score"] = venues["place_id"].map(aspect_scores)
    venues["aspect_sentiment_score"] = venues["aspect_sentiment_score"].fillna(venues["positive_sentiment_pct"])

    risk_scores = {}
    for place_id, group in top5_reviews.groupby("place_id"):
        high_risk = group.apply(
            lambda r: (r.get("review_rating", 5) <= 2) or (r.get("sentiment_label") == "NEGATIVE" and r.get("sentiment_score", 0.0) >= 0.85),
            axis=1
        )
        risk_scores[place_id] = float((high_risk.mean()) * 100.0)

    venues["negativity_risk_score"] = venues["place_id"].map(risk_scores).fillna(0.0)

    if df_aspects is not None and not df_aspects.empty:
        value_hits = df_aspects[df_aspects["aspect"] == "price or value"]
        value_scores = (value_hits.groupby("place_id")["is_positive"].mean() * 100.0).to_dict()
    else:
        value_scores = {}

    venues["value_keyword_score"] = venues["place_id"].map(value_scores).fillna(50.0)
    venues["price_factor"] = venues["price_numeric"].apply(lambda p: max(50.0, 100.0 - ((p / 500.0) * 50.0)))
    venues["price_value_score"] = (
        (0.50 * venues["positive_sentiment_pct"]) +
        (0.30 * venues["value_keyword_score"]) +
        (0.20 * venues["price_factor"])
    )

    raw_model_score = (
        (0.30 * venues["positive_sentiment_pct"]) +
        (0.25 * venues["avg_google_rating_scaled"]) +
        (0.15 * venues["volume_confidence_score"]) +
        (0.10 * venues["sentiment_momentum_score"]) +
        (0.10 * venues["aspect_sentiment_score"]) +
        (0.10 * venues["price_value_score"]) -
        (0.10 * venues["negativity_risk_score"])
    )

    tie_breaker = (venues["total_google_ratings"] % 97) * 0.001
    venues["model_score"] = np.clip(raw_model_score + tie_breaker, 0.0, 100.0).round(2)

    return venues.sort_values(by="model_score", ascending=False).reset_index(drop=True)


def _wordnet_verb_synonyms(seed_verb, sense_index):
    try:
        from nltk.corpus import wordnet as wn
        try:
            synsets = wn.synsets(seed_verb, pos=wn.VERB)
        except LookupError:
            nltk.download("wordnet", quiet=True)
            synsets = wn.synsets(seed_verb, pos=wn.VERB)
        if synsets and sense_index < len(synsets):
            return {lemma.split("_")[0].lower() for lemma in synsets[sense_index].lemma_names()}
    except Exception:
        pass
    return {seed_verb}


CONSUMPTION_VERBS = (
    _wordnet_verb_synonyms("order", 1)
    | _wordnet_verb_synonyms("taste", 2)
    | _wordnet_verb_synonyms("recommend", 0)
)
NON_FOOD_ENTITY_LABELS = {"GPE", "LOC", "ORG", "PERSON", "NORP", "FAC"}
GENERIC_FALLBACK_WORDS = {
    "place", "restaurant", "experience", "food", "menu", "price", "view",
    "service", "staff", "detail", "attention", "presentation", "wait",
    "crowd", "night", "dinner", "lunch", "visit", "quality", "thing",
    "ambiance", "atmosphere", "value", "way", "time", "portion", "spot"
}
DEFAULT_VIBE_FREQUENCIES = {
    "Welcoming": 1, "Cozy": 1, "Lively": 1, "Warm": 1,
    "Friendly": 1, "Excellent": 1, "Great": 1, "Elegant": 1
}


def _entity_token_spans(doc):
    spans = set()
    for ent in doc.ents:
        if ent.label_ in NON_FOOD_ENTITY_LABELS:
            spans.update(range(ent.start, ent.end))
    return spans


def _phrase_for_token(token, doc, entity_spans):
    for chunk in doc.noun_chunks:
        if chunk.start <= token.i < chunk.end:
            if entity_spans & set(range(chunk.start, chunk.end)):
                return None
            words = [
                t.text.lower() for t in chunk
                if t.pos_ in ("NOUN", "PROPN") and not t.is_stop and t.is_alpha
            ]
            if words and len(words) <= 3:
                return " ".join(words).title()
    return None


def extract_dish_and_vibe(texts):
    if not texts:
        return "Chef Special", "Welcoming, Cozy, Lively, Warm, Friendly, Excellent, Great, Elegant", dict(DEFAULT_VIBE_FREQUENCIES)

    doc = nlp(" ".join(texts))

    vibe_counts = Counter(
        token.lemma_.title() for token in doc
        if token.pos_ == "ADJ" and not token.is_stop and token.is_alpha and len(token.text) > 2
    )
    top_vibes = [v[0] for v in vibe_counts.most_common(10)]
    vibe_check = ", ".join(top_vibes) if top_vibes else "Welcoming, Cozy, Lively, Warm, Friendly, Excellent, Great, Elegant"
    vibe_word_frequencies = dict(vibe_counts.most_common(50)) if vibe_counts else dict(DEFAULT_VIBE_FREQUENCIES)

    entity_spans = _entity_token_spans(doc)
    primary_candidates = []
    fallback_candidates = []

    for token in doc:
        if token.dep_ in ("dobj", "obj") and token.head.lemma_.lower() in CONSUMPTION_VERBS:
            phrase = _phrase_for_token(token, doc, entity_spans)
            if phrase:
                primary_candidates.append(phrase)
        elif token.dep_ == "nsubj" and token.head.lemma_.lower() == "be":
            phrase = _phrase_for_token(token, doc, entity_spans)
            if phrase and not set(phrase.lower().split()) & GENERIC_FALLBACK_WORDS:
                fallback_candidates.append(phrase)

    dish_candidates = primary_candidates or fallback_candidates
    top_dishes = Counter(dish_candidates).most_common(1)
    famous_dish = top_dishes[0][0] if top_dishes else "Chef Special"
    return famous_dish, vibe_check, vibe_word_frequencies


def classify_review_aspects(df_reviews):
    if df_reviews.empty or "review_text" not in df_reviews.columns:
        return pd.DataFrame(columns=["place_id", "aspect", "is_positive"])

    sentences, sentence_place_ids = [], []
    for _, row in df_reviews.iterrows():
        text = str(row.get("review_text", ""))
        if not text.strip():
            continue
        for sent in nlp(text).sents:
            clean = sent.text.strip()
            if len(clean) >= 3:
                sentences.append(clean)
                sentence_place_ids.append(row["place_id"])

    if not sentences:
        return pd.DataFrame(columns=["place_id", "aspect", "is_positive"])

    aspect_predictions = aspect_classifier(sentences, ASPECT_LABELS, multi_label=True)
    sentiment_predictions = sentiment_analyzer(sentences, truncation=True, max_length=512)

    rows = []
    for place_id, aspect_pred, sentiment_pred in zip(sentence_place_ids, aspect_predictions, sentiment_predictions):
        is_positive = sentiment_pred["label"] == "POSITIVE"
        for label, score in zip(aspect_pred["labels"], aspect_pred["scores"]):
            if score >= ASPECT_CONFIDENCE_THRESHOLD:
                rows.append({"place_id": place_id, "aspect": label, "is_positive": is_positive})

    return pd.DataFrame(rows)


def generate_restaurant_insights(df_analyzed):
    if df_analyzed.empty:
        return pd.DataFrame(), {}

    if "publish_time" in df_analyzed.columns:
        df_analyzed["publish_time"] = pd.to_datetime(df_analyzed["publish_time"], errors="coerce")

    agg_list = []
    for place_id, group in df_analyzed.groupby("place_id"):
        total_reviews = len(group)
        pos_reviews = (group["sentiment_label"] == "POSITIVE").sum()
        pos_ratio = round((pos_reviews / total_reviews) * 100, 1)
        avg_rating = round(group["review_rating"].mean(), 1)

        restaurant_name = group["restaurant_name"].iloc[0]
        price_range = group["price_range"].iloc[0]
        cuisine = group["cuisine"].iloc[0]
        lat = group["latitude"].iloc[0] if "latitude" in group.columns else None
        lng = group["longitude"].iloc[0] if "longitude" in group.columns else None
        user_rating_count = group["user_rating_count"].iloc[0] if "user_rating_count" in group.columns else 0
        price_numeric = group["price_numeric"].iloc[0] if "price_numeric" in group.columns else None
        address = group["formatted_address"].iloc[0] if "formatted_address" in group.columns else None
        model_score = group["model_score"].iloc[0] if "model_score" in group.columns else 0.0

        pos_texts = group[group["sentiment_label"] == "POSITIVE"]["review_text"].tolist()
        famous_dish, vibe_check, vibe_word_frequencies = extract_dish_and_vibe(pos_texts if pos_texts else group["review_text"].tolist())

        if "publish_time" in group.columns and group["publish_time"].notna().any():
            recent_group = group.sort_values(by="publish_time", ascending=False)
        else:
            recent_group = group

        recent_3_reviews = recent_group[
            ["review_text", "publish_time", "review_rating", "sentiment_label"]
        ].head(3).to_dict(orient="records")

        pos_snippet = group[group["sentiment_label"] == "POSITIVE"]["review_text"].head(1).values
        neg_snippet = group[group["sentiment_label"] == "NEGATIVE"]["review_text"].head(1).values

        agg_list.append({
            "place_id": place_id,
            "restaurant_name": restaurant_name,
            "model_score": model_score,
            "cuisine": cuisine,
            "price_range": price_range,
            "price_numeric": price_numeric,
            "address": address,
            "avg_google_rating": avg_rating,
            "total_google_ratings": user_rating_count,
            "positive_sentiment_pct": pos_ratio,
            "reviews_analyzed": total_reviews,
            "famous_dish": famous_dish,
            "vibe_check": vibe_check,
            "vibe_word_frequencies": vibe_word_frequencies,
            "latitude": lat,
            "longitude": lng,
            "recent_3_reviews": recent_3_reviews,
            "sample_positive_review": pos_snippet[0] if len(pos_snippet) > 0 else "N/A",
            "sample_negative_review": neg_snippet[0] if len(neg_snippet) > 0 else "N/A",
        })

    df_scorecard = pd.DataFrame(agg_list)
    df_scorecard = df_scorecard.sort_values(
        by=["model_score", "positive_sentiment_pct"], ascending=[False, False]
    ).reset_index(drop=True)

    top_venue = df_scorecard.iloc[0].to_dict() if not df_scorecard.empty else {}
    return df_scorecard, top_venue


def _jsonable(obj):
    if isinstance(obj, dict):
        return {k: _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        val = float(obj)
        return None if np.isnan(val) else val
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat() if pd.notnull(obj) else None
    if isinstance(obj, float) and pd.isna(obj):
        return None
    return obj


class TTLCache:
    def __init__(self, ttl_seconds=6 * 3600):
        self.ttl_seconds = ttl_seconds
        self._store = {}

    def get(self, key):
        entry = self._store.get(key)
        if entry is None:
            return None
        value, expires_at = entry
        if time.time() > expires_at:
            del self._store[key]
            return None
        return value

    def set(self, key, value):
        self._store[key] = (value, time.time() + self.ttl_seconds)


recommendation_cache = TTLCache(ttl_seconds=6 * 3600)


def _aggregate_stats(df_scorecard):
    return {
        "restaurants_analyzed": int(len(df_scorecard)),
        "reviews_analyzed": int(df_scorecard["reviews_analyzed"].sum()),
        "avg_google_rating": round(float(df_scorecard["avg_google_rating"].mean()), 1),
        "avg_sentiment_pct": round(float(df_scorecard["positive_sentiment_pct"].mean()), 1),
        "avg_spend_per_person": round(float(df_scorecard["price_numeric"].dropna().mean()), 0)
            if df_scorecard["price_numeric"].notna().any() else None,
    }


def get_recommendations(area, cuisine="", max_budget=None, use_cache=True):
    cache_key = (area.strip().lower(), (cuisine or "").strip().lower(), max_budget)
    if use_cache:
        cached = recommendation_cache.get(cache_key)
        if cached is not None:
            return cached

    df_reviews, df_summary = get_reviews_for_area(area=area, cuisine=cuisine, max_budget=max_budget)

    if df_reviews.empty:
        result = {
            "summary": df_summary.loc[0, "transparency_note"],
            "top_pick": None,
            "restaurants": [],
            "stats": {
                "restaurants_analyzed": 0, "reviews_analyzed": 0,
                "avg_google_rating": None, "avg_sentiment_pct": None, "avg_spend_per_person": None,
            },
        }
        if use_cache:
            recommendation_cache.set(cache_key, result)
        return result

    df_analyzed = analyze_sentiment(df_reviews)
    df_aspects = classify_review_aspects(df_analyzed)
    df_scores = compute_advanced_metrics(df_analyzed, df_aspects)
    df_analyzed = df_analyzed.merge(df_scores[["place_id", "model_score"]], on="place_id", how="left")
    df_scorecard, top_venue = generate_restaurant_insights(df_analyzed)

    result = _jsonable({
        "summary": df_summary.loc[0, "transparency_note"],
        "top_pick": top_venue if top_venue else None,
        "restaurants": df_scorecard.to_dict(orient="records"),
        "stats": _aggregate_stats(df_scorecard),
    })

    if use_cache:
        recommendation_cache.set(cache_key, result)

    return result


# ==========================================
# FASTAPI APP
# ==========================================
app = FastAPI(title="Dubai Restaurant Recommendation API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET"],
    allow_headers=["*"],
)


@app.get("/api/recommend")
def recommend(
    area: str = Query(..., min_length=1, description="Required. e.g. 'Dubai Marina'"),
    cuisine: str = Query("", description="Optional cuisine filter, e.g. 'Italian'"),
    budget: float = Query(None, description="Optional max budget in AED"),
):
    try:
        return get_recommendations(area=area, cuisine=cuisine, max_budget=budget)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc))
    except RuntimeError as exc:
        raise HTTPException(status_code=502, detail=str(exc))


FRONTEND_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "frontend")


@app.get("/")
def serve_search_page():
    return FileResponse(os.path.join(FRONTEND_DIR, "search_page.html"))


@app.get("/results")
def serve_results_page():
    return FileResponse(os.path.join(FRONTEND_DIR, "results_page.html"))


@app.get("/health")
def health():
    return {"status": "ok"}


In [ ]:
import subprocess
import sys
import time
from google.colab import output

if 'server_process' in globals():
    try:
        server_process.terminate()
        server_process.wait(timeout=2)
    except Exception:
        pass

server_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(1.5)

print("🔗 Open refreshed Dashboard URL:")
print(output.eval_js("google.colab.kernel.proxyPort(8000)"))

🔗 Open refreshed Dashboard URL:
https://8000-m-s-kkb-usc1b2-5yjbiqdfz4ho-b.us-central1-2.prod.colab.dev
